In [52]:
import re

import pandas as pd

# 1. Clinical Features

In [53]:
# read in data
clinical_features = pd.read_excel("./data/matched_clinical_features.xlsx")


# infer Event 
clinical_features["event"] = clinical_features.apply(
    lambda row: True if row["overall_survival"] > row["event_free_survival"] else False,
    axis=1,
)


# select and rename columns
clinical_features = clinical_features[
    [
        "SubjectID",
         "legal_sex",
        "age_at_event_days",
        "consolidated_tumor_locations",
        "cancer_predisposition",
        "extent_of_tumor_resection",
        "chemotherapy",
        "radiation",
        "event_free_survival",
        "event",
        "Cohort",
    ]
]
clinical_features = clinical_features.rename(
    columns={
        "legal_sex": "Sex",
        "age_at_event_days": "Age at Diagnosis",
        "consolidated_tumor_locations": "Tumor Location",
        "cancer_predisposition": "NF1",
        "extent_of_tumor_resection": "Extent of Tumor Resection",
        "chemotherapy": "Chemotherapy",
        "radiation": "Radiation",
        "event_free_survival": "Progression Free Survival",
        "event": "Event",
    }
)

# set index
clinical_features.set_index("SubjectID", inplace=True)

# cache clinical features
clinical_features.to_pickle("./data/clinical_features.pkl")

print(clinical_features.shape)
print(clinical_features.columns)


(360, 10)
Index(['Sex', 'Age at Diagnosis', 'Tumor Location', 'NF1',
       'Extent of Tumor Resection', 'Chemotherapy', 'Radiation',
       'Progression Free Survival', 'Event', 'Cohort'],
      dtype='object')


# 2. Radiomic Features

In [54]:
# read in data
Resnet_features = pd.read_csv("./data/plgg_features_layer3_gap_gmp_20260308_232319.csv")

# extract subject IDs
Resnet_features["SubjectID"] = Resnet_features["SubjectID"].apply(
    lambda x: (match := re.match(r"^(C\d+|sub\d+)", x)) and match.group()
)

# filter subjects
Resnet_features = Resnet_features[
    Resnet_features["SubjectID"].isin(clinical_features.index)
]

# set index
Resnet_features.set_index("SubjectID", inplace=True)


# cache radiomic features 
Resnet_features.to_pickle("./data/Resnet_features.pkl")

print("Final ResNet feature matrix shape:", Resnet_features.shape)


Final ResNet feature matrix shape: (360, 513)


In [55]:
Resnet_features

,Session,feature_000,feature_001,feature_002,feature_003,feature_004,feature_005,feature_006,feature_007,feature_008,...,feature_502,feature_503,feature_504,feature_505,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511
SubjectID,,,,,,,,,,,,,,,,,,,,,
C1003434,4205,1.710320,0.711846,0.849767,0.989046,0.988614,0.756189,1.047282,1.169081,1.048005,...,22.112913,16.447937,19.264378,6.981818,22.527925,8.106328,18.373160,8.442207,17.608500,9.562148
C1003557,2317,1.618189,1.503287,1.571968,2.190831,1.659130,1.642314,2.112808,2.458500,2.388796,...,38.993782,25.520758,34.146175,10.964241,38.501305,11.341085,33.923172,9.395148,26.035050,10.585689
C102459,1983,1.587187,1.436070,1.463184,1.928827,1.558372,1.503438,1.876757,2.178597,2.073509,...,33.003048,21.165634,31.369024,11.610105,34.662945,17.062746,30.015888,12.738861,21.401154,16.782654
C1026189,1894,1.584791,1.694350,1.822345,2.427129,1.756830,1.849104,2.252072,2.733931,2.626046,...,51.320460,30.073833,39.243195,12.342398,45.588493,12.312659,40.418987,10.921722,25.628149,18.801386
C1026558,4971,1.733762,0.715199,0.796803,0.977295,0.977598,0.729140,1.019578,1.164458,1.016582,...,20.501625,16.087582,18.931747,6.054876,22.247177,7.513866,17.809654,6.675217,16.457157,10.009660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C972192,1923,1.713295,0.997807,1.099453,1.504884,1.228869,1.103563,1.445792,1.695149,1.565273,...,31.645138,19.757950,25.349228,11.204308,28.494892,8.432317,24.924126,9.248287,19.732052,10.183395
C972684,3226,1.644943,1.256322,1.387989,1.869427,1.462877,1.393450,1.796567,2.081498,1.991022,...,35.482475,22.410248,29.321346,13.209015,33.651220,10.541999,28.740730,9.050260,24.530592,9.822064
C973176,2509,1.672594,1.336686,1.421271,2.023916,1.510261,1.450609,1.888548,2.245399,2.128824,...,37.923900,23.607290,31.668228,11.534612,36.419990,10.689421,31.531498,8.216092,23.162392,14.309631
